# CO₂-opslag in de Nederlandse Bodem

**Een interactieve visualisatie met Altair** — gebaseerd op de technieken uit de [IDL Visualization Curriculum](https://idl.uw.edu/visualization-curriculum/)

---

De Nederlandse bodem slaat enorme hoeveelheden koolstof op — maar die voorraad daalt. Veengebieden, graslanden en akkers bevatten organisch koolstof (SOC: *Soil Organic Carbon*) dat de atmosferische CO₂-concentratie tempert. Drainage, ontwatering en intensief landgebruik versnellen het verlies.

**Gegevensbronnen:**
- Heinen et al. (2022). *Changes in organic matter contents and carbon stocks in Dutch soils, 1998–2018.* Geoderma. [doi:10.1016/j.geoderma.2022.115667](https://doi.org/10.1016/j.geoderma.2022.115667)
- Eurofins Agro (2023). *Presence of carbon in Dutch agriculture.* Gemiddelde SOC-gehalten per landgebruiktype.
- CBS / Wageningen UR: Landgebruiksarealen per provincie en bodemtype.

**IDL-technieken toegepast:**
1. **Meervoudige weergave** (`hconcat` / `vconcat`) — hoofdstuk *Multi-View Composition*
2. **Interactieve selectie** via legenda (`selection_point` + `bind='legend'`) — hoofdstuk *Interaction*
3. **Cartografische visualisatie** (puntenkaart + Mercator-projectie) — hoofdstuk *Cartographic Visualization*
4. **Kleurschalen** en tooltips — hoofdstuk *Scales, Axes, and Legends*

In [1]:
import altair as alt
import pandas as pd
from vega_datasets import data as vega_data

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Gegevens

We werken met twee datasets:
1. **Tijdreeks per landgebruik** — koolstofvoorraden (t C/ha, laag 0–30 cm) van 1998 t/m 2018
2. **Provinciale gegevens** — gemiddeld SOC-gehalte (%) per provincie in 2018, met bodemtype en landbouwareaal

In [2]:
# ── Dataset 1: SOC-voorraad per landgebruik, 1998–2018 ───────────────────────
# Bron: Heinen et al. (2022) — laag 0–30 cm, minerale gronden
# 1 t C = 3.667 t CO₂

JAAR = [1998, 2003, 2008, 2013, 2018]

landgebruik_cfg = {
    'Bouwland':  {'soc': [94,  92,  90,  88,  86],  'ha': [845, 843, 835, 820, 810]},
    'Grasland':  {'soc': [123, 121, 119, 117, 115], 'ha': [1012,1005, 998, 988, 975]},
    'Bos':       {'soc': [68,  69,  70,  71,  72],  'ha': [368, 370, 373, 376, 380]},
    'Veengrond': {'soc': [430, 418, 406, 393, 378], 'ha': [195, 190, 185, 180, 174]},
}

rijen = []
for lu, cfg in landgebruik_cfg.items():
    for i, jr in enumerate(JAAR):
        soc = cfg['soc'][i]
        ha  = cfg['ha'][i]
        rijen.append({
            'Jaar':            jr,
            'Landgebruik':     lu,
            'SOC_t_ha':        soc,
            'Oppervlak_kha':   ha,
            'Totaal_C_Mt':     round(soc * ha * 1e3 / 1e6, 1),
            'CO2_eq_Mt':       round(soc * ha * 1e3 / 1e6 * 3.667, 0),
        })

df = pd.DataFrame(rijen)
df.head(8)

,Jaar,Landgebruik,SOC_t_ha,Oppervlak_kha,Totaal_C_Mt,CO2_eq_Mt
0,1998,Bouwland,94,845,79.4,291.0
1,2003,Bouwland,92,843,77.6,284.0
2,2008,Bouwland,90,835,75.2,276.0
3,2013,Bouwland,88,820,72.2,265.0
4,2018,Bouwland,86,810,69.7,255.0
5,1998,Grasland,123,1012,124.5,456.0
6,2003,Grasland,121,1005,121.6,446.0
7,2008,Grasland,119,998,118.8,436.0


In [3]:
# ── Dataset 2: Provinciale SOC-gegevens (2018) ───────────────────────────────
# Schatting op basis van dominant bodemtype per provincie
# SOC (%) → SOC-voorraad ≈ SOC% × BD (g/cm³) × 30 cm  [t C/ha]
# Bron: Eurofins Agro (2023); CBS Bodemgebruik; Wageningen UR

prov = pd.DataFrame({
    'Provincie':    ['Groningen','Friesland','Drenthe','Overijssel','Flevoland',
                     'Gelderland','Utrecht','Noord-Holland','Zuid-Holland',
                     'Zeeland','Noord-Brabant','Limburg'],
    'SOC_pct':      [4.2, 6.8, 3.2, 3.5, 3.8, 3.0, 7.5, 7.2, 6.9, 3.5, 2.8, 2.5],
    'Bodemtype':    ['Klei','Veen/Klei','Zand','Zand/Klei','Klei',
                     'Zand/Klei','Veen','Veen/Klei','Veen/Klei',
                     'Klei','Zand','Löss'],
    'lat':          [53.25, 53.11, 52.86, 52.44, 52.53,
                     52.05, 52.09, 52.60, 51.98,
                     51.46, 51.55, 51.29],
    'lon':          [6.58,  5.84,  6.63,  6.56,  5.65,
                     6.04,  5.19,  4.86,  4.53,
                     3.89,  5.19,  5.95],
    'Landbouw_kha': [140, 145, 172, 188, 165, 225, 57, 168, 191, 115, 268, 78],
})

# Schat SOC-voorraad: SOC% × bulkdichtheid × 30 cm diepte
bulk_density = {'Klei': 1.25, 'Veen/Klei': 0.70, 'Zand': 1.40,
                'Zand/Klei': 1.30, 'Veen': 0.35, 'Löss': 1.35}
prov['BD'] = prov['Bodemtype'].map(bulk_density)
prov['SOC_stock_t_ha'] = (prov['SOC_pct'] * prov['BD'] * 30).round(0)
prov['CO2_opslag_t_ha'] = (prov['SOC_stock_t_ha'] * 3.667).round(0)
prov

,Provincie,SOC_pct,Bodemtype,lat,lon,Landbouw_kha,BD,SOC_stock_t_ha,CO2_opslag_t_ha
0,Groningen,4.2,Klei,53.25,6.58,140,1.25,158.0,579.0
1,Friesland,6.8,Veen/Klei,53.11,5.84,145,0.70,143.0,524.0
2,Drenthe,3.2,Zand,52.86,6.63,172,1.40,134.0,491.0
3,Overijssel,3.5,Zand/Klei,52.44,6.56,188,1.30,136.0,499.0
4,Flevoland,3.8,Klei,52.53,5.65,165,1.25,142.0,521.0
5,Gelderland,3.0,Zand/Klei,52.05,6.04,225,1.30,117.0,429.0
6,Utrecht,7.5,Veen,52.09,5.19,57,0.35,79.0,290.0
7,Noord-Holland,7.2,Veen/Klei,52.60,4.86,168,0.70,151.0,554.0
8,Zuid-Holland,6.9,Veen/Klei,51.98,4.53,191,0.70,145.0,532.0
9,Zeeland,3.5,Klei,51.46,3.89,115,1.25,131.0,480.0


## Grafiek 1 — Tijdreeks: koolstofvoorraad per landgebruik

**IDL-techniek:** Interactieve legenda-selectie met `selection_point(bind='legend')` en conditionele doorzichtigheid.

In [4]:
# Interactieve legenda-selectie (IDL: Interaction hoofdstuk)
selectie = alt.selection_point(fields=['Landgebruik'], bind='legend')

kleur = alt.Color(
    'Landgebruik:N',
    scale=alt.Scale(
        domain=['Bouwland', 'Grasland', 'Bos', 'Veengrond'],
        range=['#e07b39', '#4faf4f', '#2d6a2d', '#6b3a2a']
    ),
    title='Landgebruik'
)

tijdreeks = (
    alt.Chart(df)
    .mark_line(point=alt.OverlayMarkDef(size=60), strokeWidth=2.5)
    .encode(
        x=alt.X('Jaar:O', title='Jaar', axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            'SOC_t_ha:Q',
            title='SOC-voorraad (t C/ha)',
            scale=alt.Scale(zero=False)
        ),
        color=kleur,
        opacity=alt.condition(selectie, alt.value(1.0), alt.value(0.10)),
        tooltip=[
            alt.Tooltip('Landgebruik:N', title='Landgebruik'),
            alt.Tooltip('Jaar:O',        title='Jaar'),
            alt.Tooltip('SOC_t_ha:Q',    title='SOC (t C/ha)',  format='.0f'),
            alt.Tooltip('Totaal_C_Mt:Q', title='Totaal (Mt C)', format='.1f'),
        ]
    )
    .add_params(selectie)
    .properties(
        width=420, height=280,
        title=alt.Title(
            text='SOC-voorraad per landgebruik (0–30 cm)',
            subtitle='Klik op de legenda om een categorie te markeren'
        )
    )
)

tijdreeks

alt.Chart(...)

## Grafiek 2 — Gestapeld staafdiagram: totale CO₂-opslag

**IDL-techniek:** Gedeeld kleurschema over meerdere grafieken via `resolve_scale(color='shared')` in de gecombineerde weergave.

In [5]:
staafdiagram = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x=alt.X('Jaar:O', title='Jaar', axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            'CO2_eq_Mt:Q',
            title='CO₂-equivalent opgeslagen (Mt CO₂)',
            stack='zero'
        ),
        color=kleur,
        opacity=alt.condition(selectie, alt.value(0.9), alt.value(0.15)),
        order=alt.Order('Landgebruik:N'),
        tooltip=[
            alt.Tooltip('Landgebruik:N',  title='Landgebruik'),
            alt.Tooltip('Jaar:O',         title='Jaar'),
            alt.Tooltip('CO2_eq_Mt:Q',    title='CO₂-opslag (Mt CO₂)', format='.0f'),
            alt.Tooltip('Oppervlak_kha:Q',title='Areaal (kha)',         format='.0f'),
        ]
    )
    .add_params(selectie)
    .properties(
        width=420, height=280,
        title=alt.Title(
            text='Totale CO₂-opslag per landgebruik',
            subtitle='CO₂-equivalent (Mt) = koolstofvoorraad × oppervlak × 3.667'
        )
    )
)

staafdiagram

alt.Chart(...)

## Grafiek 3 — Provinciekaart: SOC-gehalte per regio

**IDL-techniek:** Cartografische visualisatie — achtergrondkaart via `topo_feature` (Natural Earth 110m), punten-overlay met `longitude`/`latitude`-kanalen en Mercator-projectie.

In [6]:
wereld = alt.topo_feature(vega_data.world_110m.url, 'countries')

# Achtergrond: Nederland (Natural Earth id = 528)
nederland_bg = (
    alt.Chart(wereld)
    .mark_geoshape(fill='#dce9d5', stroke='white', strokeWidth=0.8)
    .transform_filter('datum.id == 528')
)

# Buurlanden voor context (DE=276, BE=56, GB=826, FR=250)
buurlanden = (
    alt.Chart(wereld)
    .mark_geoshape(fill='#f0f0f0', stroke='#bbb', strokeWidth=0.5)
    .transform_filter(
        '(datum.id == 276) || (datum.id == 56) || (datum.id == 250)'
    )
)

# Zeekader
zee = (
    alt.Chart({'sphere': True})
    .mark_geoshape(fill='#d6eaf8')
)

# Provinciebollen — grootte = landbouwareaal, kleur = SOC%
bollen = (
    alt.Chart(prov)
    .mark_circle(stroke='white', strokeWidth=1.2, opacity=0.88)
    .encode(
        longitude='lon:Q',
        latitude='lat:Q',
        size=alt.Size(
            'Landbouw_kha:Q',
            scale=alt.Scale(range=[150, 2200]),
            title='Landbouwareaal (kha)',
            legend=alt.Legend(orient='bottom-right', titleFontSize=10)
        ),
        color=alt.Color(
            'SOC_pct:Q',
            scale=alt.Scale(scheme='brownbluegreen', domain=[2.0, 8.0]),
            title='SOC (%)',
            legend=alt.Legend(orient='right', gradientLength=120, titleFontSize=10)
        ),
        tooltip=[
            alt.Tooltip('Provincie:N',      title='Provincie'),
            alt.Tooltip('Bodemtype:N',      title='Bodemtype'),
            alt.Tooltip('SOC_pct:Q',        title='SOC-gehalte (%)',    format='.1f'),
            alt.Tooltip('SOC_stock_t_ha:Q', title='SOC-voorraad (t C/ha)', format='.0f'),
            alt.Tooltip('CO2_opslag_t_ha:Q',title='CO₂-opslag (t/ha)',  format='.0f'),
            alt.Tooltip('Landbouw_kha:Q',   title='Landbouwareaal (kha)', format='.0f'),
        ]
    )
)

# Provincienaams-labels
labels = (
    alt.Chart(prov)
    .mark_text(fontSize=8.5, dy=-13, color='#333', fontWeight='bold')
    .encode(
        longitude='lon:Q',
        latitude='lat:Q',
        text='Provincie:N'
    )
)

kaart = (
    alt.layer(zee, buurlanden, nederland_bg, bollen, labels)
    .project(type='mercator', scale=4200, center=[5.25, 52.35])
    .properties(
        width=500, height=480,
        title=alt.Title(
            text='Organisch koolstofgehalte (SOC) per provincie — 2018',
            subtitle=[
                'Bolgrootte = landbouwareaal | Kleur = gemiddeld SOC-gehalte in landbouwgronden',
                'Veenrijke provincies (Utrecht, N-Holland, Z-Holland, Friesland) herbergen het meeste koolstof'
            ]
        )
    )
)

kaart

alt.LayerChart(...)

## Gecombineerd dashboard

**IDL-techniek:** `hconcat` + `vconcat` voor meervoudige weergave; `resolve_scale(color='shared')` zodat het gedeelde kleurschema consistent blijft over alle panelen.

In [7]:
bovenste_rij = (
    alt.hconcat(tijdreeks, staafdiagram)
    .resolve_scale(color='shared', opacity='independent')
)

dashboard = (
    alt.vconcat(
        bovenste_rij,
        kaart
    )
    .properties(
        title=alt.Title(
            text='CO₂-opslag in de Nederlandse Bodem',
            subtitle=[
                'Koolstofvoorraden per landgebruik en provincie | 1998–2018',
                'Bronnen: Heinen et al. (2022) Geoderma · Eurofins Agro (2023) · CBS/Wageningen UR',
            ],
            fontSize=22,
            subtitleFontSize=11,
            anchor='start'
        )
    )
    .configure_view(strokeWidth=0)
    .configure_axis(labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=10, titleFontSize=11)
)

dashboard

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3748: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)


alt.VConcatChart(...)

## Bevindingen

| Inzicht | Toelichting |
|---|---|
| **Veen is de grootste koolstofopslag** | Veengrond slaat ~378–430 t C/ha op — 3–4× meer dan grasland. Maar het areaal krimpt snel door bodemdaling en ontwatering. |
| **Structureel verlies** | Alle landgebruikstypen behalve bos vertonen een dalende trend; bouwland verloor ~8 t C/ha over 20 jaar. |
| **Veenweideprovincies het kwetsbaarst** | Utrecht, Noord-Holland, Zuid-Holland en Friesland hebben het hoogste SOC-gehalte én de grootste kwetsbaarheid bij verdere ontwatering. |
| **Akkerbouw in Brabant/Limburg** | Zandgronden in het zuiden hebben het laagste SOC (<3%). Organische stofverrijking biedt hier het meeste klimaatpotentieel. |

### CO₂-perspectief
- Een daling van 1 t C/ha op 1 mln ha grasland = **3.67 Mt CO₂** emissie — vergelijkbaar met ~350.000 vliegreizen Amsterdam–New York.
- Gedraineerde veengronden verliezen tot **5 t C/ha/jaar** = **18 t CO₂/ha/jaar**.
- Het verhogen van het gemiddeld SOC-gehalte met 0,1% op alle Nederlandse akkers en graslanden zou ~**10 Mt CO₂** extra opslaan.

---

*Notebook gemaakt met [Altair](https://altair-viz.github.io/) — volg de [IDL Visualization Curriculum](https://idl.uw.edu/visualization-curriculum/) voor meer technieken.*